In [18]:
import numpy as np

#####Encoding the strings

In [19]:
data = [
    [150, 7.0, 1, 'Apple'],
    [120, 6.5, 0, 'Banana'],
    [180, 7.5, 2, 'Orange'],
    [155, 7.2, 1, 'Apple'],
    [110, 6.0, 0, 'Banana'],
    [190, 7.8, 2, 'Orange'],
    [145, 7.1, 1, 'Apple'],
    [115, 6.3, 0, 'Banana'],
]

label = {'Apple':0,'Banana':1,'Orange':2}
decode = {v:k for k,v in label.items()}

x = np.array([[row[0], row[1], row[2]] for row in data])
y = np.array([label[row[3]] for row in data],  dtype=int)

#####Min Max Normalisation

In [20]:
def min_max(x,x_min = None, x_max = None):
  if x_min is None:
    x_min = x.min(axis = 0)
  if x_max is None:
    x_max = x.max(axis=0)
  x_norm = (x - x_min) / (x_max - x_min)
  return x_norm,x_min,x_max

X,x_min,x_max = min_max(x)

print("Normalised X :", np.round(X[:3], 4))

Normalised X : [[0.5    0.5556 0.5   ]
 [0.125  0.2778 0.    ]
 [0.875  0.8333 1.    ]]


#####Train-Test Split

In [21]:
def split(X,y,testr = 0.2):
  np.random.seed(42)
  i = np.random.permutation(len(X))
  s = int(len(X)*(1-testr))
  train_i = i[:s]
  test_i = i[s:]
  return X[train_i],y[train_i],X[test_i],y[test_i]

x_train,y_train,x_test,y_test = split(X,y)


#####Euclidean Distance

In [22]:
def dist(a,b):
  return np.sqrt(np.sum((a-b)**2))

#####KNN Classifier

In [23]:
class knn:
  def __init__(self,k=3,w = False):
    self.k = k
    self.w = w

  def fit(self,x,y):
    self.x_train = x
    self.y_train = y

  def predict(self,x):
    distances = np.array([dist(x,x_tr) for x_tr in self.x_train])
    k_i = np.argsort(distances)[:self.k]
    k_l = self.y_train[k_i]
    k_dist = distances[k_i]

    if self.w:
      weights = 1/k_dist
      totals = {}
      for label, w in zip(k_l, weights):
        totals[label] = totals.get(label,0) + w
      return max(totals, key=totals.get)

    else:
      c = {}
      for label in k_l:
        c[label] = c.get(label,0) + 1
      return max(c, key=c.get)

  def pred(self,x_test):
    return np.array([self.predict(x) for x in x_test])

  def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)



#####Test

In [24]:
test = np.array([
    [118, 6.2, 0],
    [160, 7.3, 1],
    [185, 7.7, 2],
], dtype=float)

test_data,_,_ = min_max(test,x_min,x_max)

k = knn(k=3,w = False)
k.fit(x_train,y_train)
predictions = k.pred(test_data)

expected = ['Banana', 'Apple', 'Orange']
for i, pred in enumerate(predictions):
    status = "CORRECT" if decode[pred] == expected[i] else "WRONG"
    print(f"  [{status}] Sample {i+1}: Predicted = {decode[pred]:<8}  Expected = {expected[i]}")

  [CORRECT] Sample 1: Predicted = Banana    Expected = Banana
  [WRONG] Sample 2: Predicted = Orange    Expected = Apple
  [CORRECT] Sample 3: Predicted = Orange    Expected = Orange
